# AliPOS table-booking discovery

This notebook is read-only. It uses official AliPOS credentials only for authentication. Deployed restaurant reads are user-authorized only when both `ALLOW_LIVE_ALIPOS_READS=1` and `ALLOW_DEPLOYED_ALIPOS_READS=1`; dry runs make no network requests.

In [1]:
from __future__ import annotations

import json
import math
import os
import re
from pathlib import Path
from typing import Any, Mapping, Optional, Tuple
from urllib.parse import urlsplit


LIVE_FLAG_NAME = "ALLOW_LIVE_ALIPOS_READS"
DEPLOYED_LIVE_FLAG_NAME = "ALLOW_DEPLOYED_ALIPOS_READS"
MAX_TIMEOUT_SECONDS = 15
MINIMUM_INTERVAL_SECONDS = 0.25
MAX_PROBE_REQUESTS = 120
UUID_PATTERN = r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}"
UUID_RE = re.compile(rf"^{UUID_PATTERN}$")


class ProbeSafetyError(RuntimeError):
    pass


class LiveProbesDisabled(ProbeSafetyError):
    pass


def find_repo_root(start: Optional[Path] = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / ".env").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise ProbeSafetyError("Repository root with .env and notebooks was not found")


def read_dotenv(path: Path) -> dict:
    values = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in {"'", '"'}:
            value = value[1:-1]
        values[key] = value
    return values


def _stream_output_text(cell: Mapping[str, Any]) -> str:
    parts = []
    for output in cell.get("outputs", []):
        text = output.get("text", "")
        if isinstance(text, list):
            parts.extend(str(item) for item in text)
        elif text:
            parts.append(str(text))
    return "".join(parts)


def extract_dummy_identifiers(source_notebook: Path) -> Tuple[str, Tuple[str, ...]]:
    document = json.loads(source_notebook.read_text(encoding="utf-8"))
    cells = document.get("cells", [])
    if len(cells) < 10:
        raise ProbeSafetyError("Source notebook does not contain cells 4 and 10")

    config_output = _stream_output_text(cells[3])
    restaurant_match = re.search(rf"Restaurant ID:\s*({UUID_PATTERN})", config_output)
    if restaurant_match is None:
        raise ProbeSafetyError("Dummy restaurant ID was not found in source cell 4")

    created_output = _stream_output_text(cells[9])
    marker = "ID созданных заказов / Yaratilgan buyurtma ID lari:"
    if marker not in created_output:
        raise ProbeSafetyError("Dummy order marker was not found in source cell 10")
    tail = created_output.split(marker, 1)[1].lstrip()
    try:
        order_map, _ = json.JSONDecoder().raw_decode(tail)
    except (TypeError, ValueError) as exc:
        raise ProbeSafetyError("Dummy order map in source cell 10 is invalid") from exc

    order_ids = tuple(order_map.get(name, "") for name in ("cash", "online-order"))
    if any(not UUID_RE.fullmatch(value) for value in order_ids):
        raise ProbeSafetyError("Source cell 10 does not contain two valid dummy order IDs")
    return restaurant_match.group(1), order_ids


def build_probe_config(
    repo_root: Path,
    environ: Optional[Mapping[str, str]] = None,
) -> dict:
    dotenv = read_dotenv(repo_root / ".env")
    process = dict(os.environ if environ is None else environ)

    def configured(name: str) -> str:
        return process.get(name) or dotenv.get(name, "")

    dummy_restaurant_id, dummy_order_ids = extract_dummy_identifiers(
        repo_root / "notebooks" / "alipos_support_report_ru_uz.ipynb"
    )
    return {
        "base_url": configured("ALIPOS_API_BASE_URL").rstrip("/"),
        "client_id": configured("ALIPOS_API_CLIENT_ID"),
        "client_secret": configured("ALIPOS_API_CLIENT_SECRET"),
        "deployed_restaurant_id": configured("ALIPOS_RESTAURANT_ID"),
        "dummy_restaurant_id": dummy_restaurant_id,
        "dummy_order_ids": dummy_order_ids,
        "live_enabled": process.get(LIVE_FLAG_NAME) == "1",
        "deployed_reads_enabled": process.get(DEPLOYED_LIVE_FLAG_NAME) == "1",
        "timeout_seconds": 15,
        "minimum_interval_seconds": 0.25,
        "max_requests": 120,
    }


def validate_probe_config(
    config: Mapping[str, Any],
    require_live: bool = True,
) -> None:
    required = (
        "base_url",
        "client_id",
        "client_secret",
        "deployed_restaurant_id",
        "dummy_restaurant_id",
    )
    missing = [name for name in required if not str(config.get(name, "")).strip()]
    if missing:
        raise ProbeSafetyError("Required AliPOS configuration is missing")

    parsed = urlsplit(str(config["base_url"]))
    hostname = (parsed.hostname or "").lower()
    if parsed.scheme != "https" or not hostname:
        raise ProbeSafetyError("AliPOS base URL must use HTTPS")
    if hostname != "alipos.uz" and not hostname.endswith(".alipos.uz"):
        raise ProbeSafetyError("Configured host is not an AliPOS host")
    if parsed.username or parsed.password:
        raise ProbeSafetyError("AliPOS base URL must not contain credentials")

    deployed_id = str(config["deployed_restaurant_id"])
    dummy_id = str(config["dummy_restaurant_id"])
    if not UUID_RE.fullmatch(deployed_id) or not UUID_RE.fullmatch(dummy_id):
        raise ProbeSafetyError("Restaurant IDs must be UUIDs")
    target_is_deployed = deployed_id.casefold() == dummy_id.casefold()
    if target_is_deployed and require_live and not bool(config.get("deployed_reads_enabled")):
        raise ProbeSafetyError("Deployed restaurant reads require explicit authorization")

    order_ids = tuple(config.get("dummy_order_ids", ()))
    if len(order_ids) != 2 or any(not UUID_RE.fullmatch(str(value)) for value in order_ids):
        raise ProbeSafetyError("Exactly two valid dummy order IDs are required")

    timeout = config.get("timeout_seconds")
    if (
        isinstance(timeout, bool)
        or not isinstance(timeout, (int, float))
        or not math.isfinite(float(timeout))
        or not 0 < timeout <= MAX_TIMEOUT_SECONDS
    ):
        raise ProbeSafetyError("timeout_seconds must be between 0 and 15")

    interval = config.get("minimum_interval_seconds")
    if (
        isinstance(interval, bool)
        or not isinstance(interval, (int, float))
        or not math.isfinite(float(interval))
        or interval < MINIMUM_INTERVAL_SECONDS
    ):
        raise ProbeSafetyError("minimum_interval_seconds must be at least 0.25")

    max_requests = config.get("max_requests")
    if (
        isinstance(max_requests, bool)
        or not isinstance(max_requests, int)
        or not 0 < max_requests <= MAX_PROBE_REQUESTS
    ):
        raise ProbeSafetyError("max_requests must be an integer between 1 and 120")
    if require_live and not bool(config.get("live_enabled")):
        raise LiveProbesDisabled(f"Set {LIVE_FLAG_NAME}=1 only for the approved live run")


In [2]:
import hashlib
from typing import Sequence


SENSITIVE_KEY_TOKENS = {
    "address",
    "authorization",
    "card",
    "coordinate",
    "coordinates",
    "credential",
    "credentials",
    "cvv",
    "email",
    "lat",
    "latitude",
    "lng",
    "longitude",
    "number",
    "otp",
    "pan",
    "payment",
    "phone",
    "secret",
    "token",
}
SENSITIVE_KEY_COMPOUNDS = {
    "cardnumber",
    "clientid",
    "clientname",
    "clientsecret",
    "customername",
    "paymentinfo",
}
EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
UZ_PHONE_RE = re.compile(r"\+?998[0-9 ()-]{9,16}")
BEARER_RE = re.compile(r"(?i)\bBearer\s+[A-Za-z0-9._~-]+")
UUID_TEXT_RE = re.compile(UUID_PATTERN)


def fingerprint_identifier(value: str) -> str:
    digest = hashlib.sha256(value.encode("utf-8")).hexdigest()[:10]
    return f"id:{digest}"


def sanitize_text(text: str) -> str:
    value = UUID_TEXT_RE.sub(lambda match: fingerprint_identifier(match.group(0)), str(text))
    value = BEARER_RE.sub("Bearer [REDACTED]", value)
    value = EMAIL_RE.sub("[REDACTED_EMAIL]", value)
    value = UZ_PHONE_RE.sub("[REDACTED_PHONE]", value)
    return value[:500]


def _is_sensitive_key(key: str) -> bool:
    separated = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", str(key))
    parts = [part.casefold() for part in re.findall(r"[A-Za-z0-9]+", separated)]
    compact = "".join(parts)
    return any(part in SENSITIVE_KEY_TOKENS for part in parts) or compact in SENSITIVE_KEY_COMPOUNDS


def redact_value(value: Any, key: str = "") -> Any:
    if key and _is_sensitive_key(key):
        return "[REDACTED]"
    if isinstance(value, Mapping):
        return {
            str(item_key): redact_value(item_value, str(item_key))
            for item_key, item_value in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [redact_value(item) for item in value]
    if isinstance(value, str):
        if UUID_RE.fullmatch(value):
            return fingerprint_identifier(value)
        return sanitize_text(value)
    return value


def _case_insensitive_get(mapping: Mapping[str, Any], name: str, default: Any) -> Any:
    wanted = name.casefold()
    for key, value in mapping.items():
        if str(key).casefold() == wanted:
            return value
    return default


def _safe_shape_key(key: Any) -> str:
    return sanitize_text(str(key))


def _safe_collection_metadata(items: Sequence[Any]) -> dict:
    item_fields = []
    if items and isinstance(items[0], Mapping):
        item_fields = sorted(_safe_shape_key(key) for key in items[0].keys())
    return {"count": len(items), "item_fields": item_fields}


def _safe_service_percent(value: Any) -> Any:
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        return "[REDACTED]"
    return value


def _safe_hall_table_preview(payload: Mapping[str, Any]) -> dict:
    halls = _case_insensitive_get(payload, "halls", [])
    tables = _case_insensitive_get(payload, "tables", [])
    safe_halls = []
    safe_tables = []
    for hall in halls if isinstance(halls, list) else []:
        if not isinstance(hall, Mapping):
            continue
        safe_halls.append(
            {
                "id": redact_value(_case_insensitive_get(hall, "id", "")),
                "title": sanitize_text(str(_case_insensitive_get(hall, "title", ""))),
                "servicePercent": _safe_service_percent(
                    _case_insensitive_get(hall, "servicePercent", None)
                ),
            }
        )
    for table in tables if isinstance(tables, list) else []:
        if not isinstance(table, Mapping):
            continue
        safe_tables.append(
            {
                "id": redact_value(_case_insensitive_get(table, "id", "")),
                "title": sanitize_text(str(_case_insensitive_get(table, "title", ""))),
                "hallId": redact_value(_case_insensitive_get(table, "hallId", "")),
            }
        )
    return {"halls": safe_halls, "tables": safe_tables}


def summarize_payload(payload: Any, route_name: str = "") -> dict:
    if isinstance(payload, Mapping):
        collections = {
            _safe_shape_key(key): _safe_collection_metadata(value)
            for key, value in payload.items()
            if isinstance(value, list)
        }
        summary = {
            "type": "object",
            "fields": sorted(_safe_shape_key(key) for key in payload.keys()),
            "collections": collections,
        }
        if route_name == "halls_and_tables":
            summary["preview"] = _safe_hall_table_preview(payload)
        return summary
    if isinstance(payload, list):
        return {
            "type": "array",
            "count": len(payload),
            "item_fields": (
                sorted(_safe_shape_key(key) for key in payload[0].keys())
                if payload and isinstance(payload[0], Mapping)
                else []
            ),
        }
    if payload is None:
        return {"type": "none"}
    return {"type": type(payload).__name__}


def extract_hall_table_ids(payload: Any) -> Tuple[Optional[str], Optional[str]]:
    if not isinstance(payload, Mapping):
        return None, None
    halls = _case_insensitive_get(payload, "halls", [])
    tables = _case_insensitive_get(payload, "tables", [])

    def first_id(items: Any) -> Optional[str]:
        if not isinstance(items, list):
            return None
        for item in items:
            if not isinstance(item, Mapping):
                continue
            value = _case_insensitive_get(item, "id", "")
            if isinstance(value, str) and UUID_RE.fullmatch(value):
                return value
        return None

    return first_id(halls), first_id(tables)


def classify_result(result: Mapping[str, Any]) -> str:
    status = result.get("status")
    content_type = str(result.get("content_type", "")).casefold()
    allow = {part.strip().upper() for part in str(result.get("allow", "")).split(",") if part.strip()}
    if status in (401, 403):
        return "unauthorized/forbidden"
    if status in (400, 409, 422):
        return "invalid test data"
    if status == 404:
        return "unsupported"
    if status == 405:
        return "ambiguous" if {"GET", "OPTIONS"} & allow else "unsupported"
    if isinstance(status, int) and 200 <= status < 300:
        if "json" in content_type and result.get("payload") is not None:
            return "confirmed"
        return "ambiguous"
    return "ambiguous"


def summarize_result(result: Mapping[str, Any]) -> dict:
    error = result.get("error")
    return {
        "name": str(result.get("name", "")),
        "family": str(result.get("family", "")),
        "method": str(result.get("method", "")),
        "path": sanitize_text(str(result.get("path", ""))),
        "status": result.get("status"),
        "classification": classify_result(result),
        "latency_ms": result.get("latency_ms"),
        "content_type": sanitize_text(str(result.get("content_type", ""))),
        "allow": sanitize_text(str(result.get("allow", ""))),
        "shape": summarize_payload(result.get("payload"), str(result.get("name", ""))),
        "error": "[REDACTED]" if error else "",
    }


In [3]:
MAX_TOTAL_REQUESTS = 120
OPTIONS_RESERVE = 12
BOOKING_TERMS = (
    "tableBooking",
    "table-booking",
    "tableReservation",
    "table-reservation",
    "booking",
    "bookings",
    "reservation",
    "reservations",
    "availability",
    "tableAvailability",
    "table-availability",
    "bookingAvailability",
    "booking-availability",
    "table",
    "tables",
    "hall",
    "halls",
    "floor",
    "floors",
    "reserve",
    "reserves",
    "bron",
)
ID_SCOPED_TERMS = BOOKING_TERMS


def _route(name: str, family: str, path: str, method: str = "GET") -> dict:
    return {"name": name, "family": family, "method": method, "path": path}


def _deduplicate_routes(routes: Sequence[Mapping[str, str]]) -> list:
    selected = []
    seen = set()
    for route in routes:
        key = (str(route["method"]), str(route["path"]))
        if key in seen:
            continue
        seen.add(key)
        selected.append(dict(route))
    return selected


def build_documented_routes(restaurant_id: str) -> list:
    return [
        _route("restaurants", "documented", "/restaurants"),
        _route("payment_methods", "documented", "/api/Integration/v1/paymentMethod/all"),
        _route(
            "menu_composition",
            "documented",
            f"/api/Integration/v1/menu/{restaurant_id}/composition",
        ),
        _route(
            "halls_and_tables",
            "documented",
            f"/api/Integration/v1/restaurant/{restaurant_id}/halls-and-tables",
        ),
    ]


def build_order_routes(order_ids: Sequence[str]) -> list:
    routes = []
    for index, order_id in enumerate(order_ids, start=1):
        routes.append(
            _route(
                f"dummy_order_{index}",
                "order_read",
                f"/api/Integration/v1/order/{order_id}",
            )
        )
        routes.append(
            _route(
                f"dummy_order_status_{index}",
                "order_read",
                f"/api/Integration/v1/order/{order_id}/status",
            )
        )
    return routes


def build_booking_routes(
    restaurant_id: str,
    hall_id: Optional[str] = None,
    table_id: Optional[str] = None,
) -> list:
    routes = []
    for term in BOOKING_TERMS:
        routes.extend(
            [
                _route(f"{term}_top", "top_level", f"/api/Integration/v1/{term}"),
                _route(
                    f"{term}_restaurant_scoped",
                    "restaurant_scoped",
                    f"/api/Integration/v1/restaurant/{restaurant_id}/{term}",
                ),
                _route(
                    f"{term}_restaurant_argument",
                    "argument",
                    f"/api/Integration/v1/{term}/{restaurant_id}",
                ),
                _route(
                    f"{term}_menu_scoped",
                    "menu_scoped",
                    f"/api/Integration/v1/menu/{restaurant_id}/{term}",
                ),
            ]
        )

    for term in ID_SCOPED_TERMS:
        if table_id:
            routes.extend(
                [
                    _route(
                        f"{term}_table_argument",
                        "id_scoped",
                        f"/api/Integration/v1/{term}/{table_id}",
                    ),
                    _route(
                        f"{term}_restaurant_table",
                        "id_scoped",
                        f"/api/Integration/v1/restaurant/{restaurant_id}/{term}/{table_id}",
                    ),
                ]
            )
        if hall_id:
            routes.extend(
                [
                    _route(
                        f"{term}_hall_argument",
                        "id_scoped",
                        f"/api/Integration/v1/{term}/{hall_id}",
                    ),
                    _route(
                        f"{term}_restaurant_hall",
                        "id_scoped",
                        f"/api/Integration/v1/restaurant/{restaurant_id}/{term}/{hall_id}",
                    ),
                ]
            )
    return _deduplicate_routes(routes)


def select_get_routes(
    candidates: Sequence[Mapping[str, str]],
    completed_count: int,
) -> list:
    available = max(0, MAX_TOTAL_REQUESTS - OPTIONS_RESERVE - completed_count)
    return [dict(route) for route in candidates[:available]]


def build_option_routes(
    get_results: Sequence[Mapping[str, Any]],
    completed_count: int,
) -> list:
    available = max(0, MAX_TOTAL_REQUESTS - completed_count)
    method_failures = [result for result in get_results if result.get("status") == 405]
    representatives = []
    represented_families = set()
    for result in get_results:
        family = str(result.get("family", ""))
        if result.get("status") != 404 or family in represented_families:
            continue
        represented_families.add(family)
        representatives.append(result)

    routes = [
        _route(
            f"options_{result.get('name', 'route')}",
            str(result.get("family", "")),
            str(result.get("path", "")),
            method="OPTIONS",
        )
        for result in (*method_failures, *representatives)
    ]
    return _deduplicate_routes(routes)[:available]


In [4]:
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode, urljoin
from urllib.request import HTTPRedirectHandler, Request, build_opener
import time


ALLOWED_DISCOVERY_METHODS = frozenset({"GET", "OPTIONS"})
MAX_RESPONSE_BYTES = 5_000_000


class UnsafeMethodError(ProbeSafetyError):
    pass


class UnsafeRedirectError(ProbeSafetyError):
    pass


class AuthenticationError(ProbeSafetyError):
    pass


class RequestBudgetExceeded(ProbeSafetyError):
    pass


class NoRedirectHandler(HTTPRedirectHandler):
    def redirect_request(self, request, file_pointer, code, message, headers, new_url):
        raise UnsafeRedirectError("Redirects are disabled for AliPOS discovery")


def _header_value(headers: Any, name: str) -> str:
    if headers is None:
        return ""
    value = headers.get(name, "")
    return str(value or "")


def _read_limited_body(response: Any) -> bytes:
    body = response.read(MAX_RESPONSE_BYTES + 1)
    return body[:MAX_RESPONSE_BYTES]


def _decode_payload(body: bytes, content_type: str) -> Tuple[Any, str]:
    if not body:
        return None, ""
    text = body.decode("utf-8", errors="replace")
    if "json" in content_type.casefold() or text.lstrip().startswith(("{", "[")):
        try:
            return json.loads(text), ""
        except ValueError:
            return None, "Malformed JSON response"
    return None, "Non-JSON response body suppressed"


class SafeAliPOSClient:
    def __init__(
        self,
        config: Mapping[str, Any],
        opener: Any = None,
        sleep_fn: Any = time.sleep,
        monotonic_fn: Any = time.monotonic,
    ) -> None:
        validate_probe_config(config, require_live=True)
        self._config = dict(config)
        self._origin = urlsplit(str(config["base_url"]))
        self._opener = opener or build_opener(NoRedirectHandler())
        self._sleep = sleep_fn
        self._monotonic = monotonic_fn
        self._access_token = None
        self._request_count = 0
        self._rate_limited = False
        self._last_request_at = None

    @property
    def request_count(self) -> int:
        return self._request_count

    @property
    def rate_limited(self) -> bool:
        return self._rate_limited

    def _latch_rate_limit(self, status: int) -> None:
        if status == 429:
            self._rate_limited = True

    def _build_url(self, path: str) -> str:
        if not path.startswith("/"):
            raise ProbeSafetyError("Probe path must start with a slash")
        url = urljoin(f"{self._config['base_url']}/", path.lstrip("/"))
        parsed = urlsplit(url)
        if (parsed.scheme, parsed.netloc) != (self._origin.scheme, self._origin.netloc):
            raise UnsafeRedirectError("Probe URL origin differs from AliPOS origin")
        return url

    def _assert_final_origin(self, response: Any) -> None:
        final_url = str(response.geturl())
        parsed = urlsplit(final_url)
        if (parsed.scheme, parsed.netloc) != (self._origin.scheme, self._origin.netloc):
            raise UnsafeRedirectError("Response URL origin differs from AliPOS origin")

    def _throttle(self) -> None:
        now = self._monotonic()
        if self._last_request_at is not None:
            elapsed = now - self._last_request_at
            remaining = float(self._config["minimum_interval_seconds"]) - elapsed
            if remaining > 0:
                self._sleep(remaining)
        self._last_request_at = self._monotonic()

    def authenticate(self) -> None:
        if self._rate_limited:
            raise RequestBudgetExceeded("AliPOS discovery stopped after rate limit")
        body = urlencode(
            {
                "client_id": self._config["client_id"],
                "client_secret": self._config["client_secret"],
                "grant_type": "client_credentials",
            }
        ).encode("utf-8")
        request = Request(
            self._build_url("/security/oauth/token"),
            data=body,
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            method="POST",
        )
        try:
            with self._opener.open(
                request,
                timeout=int(self._config["timeout_seconds"]),
            ) as response:
                self._assert_final_origin(response)
                status = int(getattr(response, "status", 200))
                self._latch_rate_limit(status)
                payload = json.loads(_read_limited_body(response).decode("utf-8"))
        except HTTPError as exc:
            self._assert_final_origin(exc)
            self._latch_rate_limit(int(exc.code))
            raise AuthenticationError(f"AliPOS authentication returned HTTP {exc.code}") from exc
        except (URLError, ValueError, KeyError) as exc:
            raise AuthenticationError("AliPOS authentication did not return a usable token") from exc
        if not 200 <= status < 300:
            raise AuthenticationError(f"AliPOS authentication returned HTTP {status}")
        token = payload.get("access_token") if isinstance(payload, Mapping) else None
        if not isinstance(token, str) or not token:
            raise AuthenticationError("AliPOS authentication response omitted access_token")
        self._access_token = token

    def request(self, method: str, path: str, name: str, family: str) -> dict:
        normalized_method = method.upper()
        if normalized_method not in ALLOWED_DISCOVERY_METHODS:
            raise UnsafeMethodError(f"HTTP method {normalized_method} is blocked")
        if self._rate_limited:
            raise RequestBudgetExceeded("AliPOS discovery stopped after rate limit")
        if not self._access_token:
            raise AuthenticationError("Authenticate before running discovery requests")
        if self._request_count >= int(self._config["max_requests"]):
            raise RequestBudgetExceeded("AliPOS discovery request budget is exhausted")

        url = self._build_url(path)
        self._throttle()
        request = Request(
            url,
            headers={
                "Accept": "application/json",
                "Authorization": f"Bearer {self._access_token}",
            },
            method=normalized_method,
        )
        self._request_count += 1
        started_at = self._monotonic()
        status = None
        headers = {}
        payload = None
        error = ""
        try:
            with self._opener.open(
                request,
                timeout=int(self._config["timeout_seconds"]),
            ) as response:
                self._assert_final_origin(response)
                status = int(getattr(response, "status", 200))
                self._latch_rate_limit(status)
                headers = response.headers
                content_type = _header_value(headers, "Content-Type")
                payload, error = _decode_payload(_read_limited_body(response), content_type)
        except HTTPError as exc:
            self._assert_final_origin(exc)
            status = int(exc.code)
            self._latch_rate_limit(status)
            headers = exc.headers
            content_type = _header_value(headers, "Content-Type")
            payload, error = _decode_payload(_read_limited_body(exc), content_type)
        except URLError as exc:
            content_type = ""
            error = sanitize_text(str(exc.reason))

        result = {
            "name": name,
            "family": family,
            "method": normalized_method,
            "path": path,
            "status": status,
            "latency_ms": round((self._monotonic() - started_at) * 1000, 1),
            "content_type": content_type,
            "allow": _header_value(headers, "Allow"),
            "payload": payload,
            "error": error,
        }
        result["classification"] = classify_result(result)
        return result


def run_routes(client: SafeAliPOSClient, routes: Sequence[Mapping[str, str]]) -> list:
    results = []
    for route in routes:
        try:
            result = client.request(
                str(route["method"]),
                str(route["path"]),
                str(route["name"]),
                str(route["family"]),
            )
        except RequestBudgetExceeded:
            break
        results.append(result)
        if result.get("status") == 429:
            break
    return results


In [5]:
from collections import Counter


def render_markdown_report(
    results: Sequence[Mapping[str, Any]],
    dry_routes: Sequence[Mapping[str, str]] = (),
) -> str:
    lines = ["# AliPOS Table Booking Discovery", ""]
    if dry_routes and not results:
        lines.extend(
            [
                "**Mode:** dry run; no AliPOS API request was sent.",
                "",
                f"Planned GET routes: {len(dry_routes)}",
                "",
                "Live execution requires both explicit AliPOS read flags.",
            ]
        )
        return "\n".join(lines)

    summaries = [summarize_result(result) for result in results]
    counts = Counter(summary["classification"] for summary in summaries)
    lines.extend(["**Mode:** live read-only probe.", "", "## Classification counts", ""])
    for name in (
        "confirmed",
        "unsupported",
        "unauthorized/forbidden",
        "invalid test data",
        "ambiguous",
        "skipped",
    ):
        lines.append(f"- {name}: {counts.get(name, 0)}")

    lines.extend(["", "## Route results", ""])
    lines.append("| Method | Route | Status | Classification | Shape |")
    lines.append("|---|---|---:|---|---|")
    for summary in summaries:
        shape = json.dumps(summary["shape"], ensure_ascii=False, sort_keys=True)
        lines.append(
            "| {method} | `{path}` | {status} | {classification} | `{shape}` |".format(
                method=summary["method"],
                path=summary["path"].replace("|", "\\|"),
                status=summary["status"] if summary["status"] is not None else "network-error",
                classification=summary["classification"],
                shape=shape.replace("|", "\\|")[:500],
            )
        )

    confirmed_booking = [
        summary
        for summary in summaries
        if summary["classification"] == "confirmed"
        and summary["method"] == "GET"
        and summary["family"] not in {"documented", "order_read"}
    ]
    lines.extend(["", "## Conclusion", ""])
    if confirmed_booking:
        lines.append("At least one non-documented booking-related read route returned usable JSON.")
    else:
        lines.append("No native booking or availability read route was confirmed by this run.")
    lines.append("Halls and tables alone do not confirm reservation support.")
    return "\n".join(lines)


In [6]:
ROOT = find_repo_root()
CONFIG = build_probe_config(ROOT)
validate_probe_config(CONFIG, require_live=False)
print("Configuration loaded; deployed target requires the second live-read flag.")
print("Live reads enabled:", CONFIG["live_enabled"])
print("Deployed reads enabled:", CONFIG["deployed_reads_enabled"])


Configuration loaded; deployed target requires the second live-read flag.
Live reads enabled: True
Deployed reads enabled: True


In [7]:
_synthetic_config = dict(CONFIG)
_synthetic_config.update(
    {
        "base_url": "https://web.alipos.uz",
        "client_id": "synthetic-client-id",
        "client_secret": "synthetic-client-secret",
        "deployed_restaurant_id": "22222222-2222-4222-8222-222222222222",
        "dummy_restaurant_id": "11111111-1111-4111-8111-111111111111",
        "dummy_order_ids": (
            "33333333-3333-4333-8333-333333333333",
            "44444444-4444-4444-8444-444444444444",
        ),
        "live_enabled": True,
    }
)
validate_probe_config(_synthetic_config)
assert redact_value({"phoneNumber": "+998901234567"})["phoneNumber"] == "[REDACTED]"
assert all(route["method"] == "GET" for route in build_documented_routes(_synthetic_config["dummy_restaurant_id"]))
print("Offline safety self-checks passed.")


Offline safety self-checks passed.


In [8]:
CLIENT = None
RESULTS = []
DOCUMENTED_ROUTES = build_documented_routes(CONFIG["dummy_restaurant_id"])
ORDER_ROUTES = build_order_routes(CONFIG["dummy_order_ids"])
if CONFIG["live_enabled"]:
    validate_probe_config(CONFIG, require_live=True)
    CLIENT = SafeAliPOSClient(CONFIG)
    CLIENT.authenticate()
    print("AliPOS authentication succeeded; token output is suppressed.")
else:
    print("Dry run: authentication and all network requests are skipped.")


AliPOS authentication succeeded; token output is suppressed.


In [9]:
HALL_ID = None
TABLE_ID = None
if CLIENT is not None:
    baseline_results = run_routes(CLIENT, DOCUMENTED_ROUTES)
    RESULTS.extend(baseline_results)
    halls_result = next(
        (result for result in baseline_results if result["name"] == "halls_and_tables"),
        None,
    )
    if halls_result is not None:
        HALL_ID, TABLE_ID = extract_hall_table_ids(halls_result.get("payload"))
    RESULTS.extend(run_routes(CLIENT, ORDER_ROUTES))
    print("Documented and dummy-order GET probes completed.")
else:
    print("Dry run documented route names:", [route["name"] for route in DOCUMENTED_ROUTES])
    print("Dry run dummy-order route count:", len(ORDER_ROUTES))


Documented and dummy-order GET probes completed.


In [10]:
BOOKING_CANDIDATES = build_booking_routes(
    CONFIG["dummy_restaurant_id"],
    hall_id=HALL_ID,
    table_id=TABLE_ID,
)
if CLIENT is not None:
    selected_gets = select_get_routes(BOOKING_CANDIDATES, CLIENT.request_count)
    booking_results = run_routes(CLIENT, selected_gets)
    RESULTS.extend(booking_results)
    option_routes = build_option_routes(booking_results, CLIENT.request_count)
    RESULTS.extend(run_routes(CLIENT, option_routes))
    print("Booking discovery completed within request budget:", CLIENT.request_count)
else:
    selected_gets = select_get_routes(
        BOOKING_CANDIDATES,
        len(DOCUMENTED_ROUTES) + len(ORDER_ROUTES),
    )
    print("Dry run booking GET route count:", len(selected_gets))


Booking discovery completed within request budget: 113


In [11]:
from IPython.display import Markdown, display


DRY_ROUTES = [] if CLIENT is not None else [*DOCUMENTED_ROUTES, *ORDER_ROUTES, *selected_gets]
REPORT = render_markdown_report(RESULTS, dry_routes=DRY_ROUTES)
display(Markdown(REPORT))


# AliPOS Table Booking Discovery

**Mode:** live read-only probe.

## Classification counts

- confirmed: 9
- unsupported: 104
- unauthorized/forbidden: 0
- invalid test data: 0
- ambiguous: 0
- skipped: 0

## Route results

| Method | Route | Status | Classification | Shape |
|---|---|---:|---|---|
| GET | `/restaurants` | 200 | confirmed | `{"collections": {"places": {"count": 1, "item_fields": ["address", "id", "title"]}}, "fields": ["places"], "type": "object"}` |
| GET | `/api/Integration/v1/paymentMethod/all` | 200 | confirmed | `{"count": 4, "item_fields": ["id", "isExternallyFiscalized", "title"], "type": "array"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/composition` | 200 | confirmed | `{"collections": {"categories": {"count": 4, "item_fields": ["id", "name", "schedules", "sortOrder"]}, "items": {"count": 54, "item_fields": ["categoryId", "description", "id", "images", "measure", "measureUnit", "modifierGroups", "name", "price", "serviceCodesUz", "sortOrder", "vat"]}}, "fields": ["categories", "items", "lastChange", "schedules"], "type": "object"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/halls-and-tables` | 200 | confirmed | `{"collections": {"halls": {"count": 1, "item_fields": ["id", "servicePercent", "title"]}, "tables": {"count": 29, "item_fields": ["hallId", "id", "title"]}}, "fields": ["halls", "tables"], "preview": {"halls": [{"id": "id:40aa4c5bcb", "servicePercent": 10.0, "title": "Zal"}], "tables": [{"hallId": "id:40aa4c5bcb", "id": "id:b7fef41433", "title": "Stoll 1"}, {"hallId": "id:40aa4c5bcb", "id": "id:216ac66fcb", "title": "Stoll 28"}, {"hallId": "id:40aa4c5bcb", "id": "id:5dfa7ffea4", "title": "Stoll ` |
| GET | `/api/Integration/v1/order/id:37eb66540b` | 200 | confirmed | `{"collections": {"items": {"count": 1, "item_fields": ["id", "modifications", "name", "price", "promos", "quantity"]}}, "fields": ["comment", "deliveryInfo", "discriminator", "eatsId", "id", "items", "orderNumber", "paymentInfo", "persons", "platform", "restaurantId", "status"], "type": "object"}` |
| GET | `/api/Integration/v1/order/id:37eb66540b/status` | 200 | confirmed | `{"collections": {}, "fields": ["comment", "status", "updatedAt"], "type": "object"}` |
| GET | `/api/Integration/v1/order/id:8d853bd485` | 200 | confirmed | `{"collections": {"items": {"count": 1, "item_fields": ["id", "modifications", "name", "price", "promos", "quantity"]}}, "fields": ["comment", "deliveryInfo", "discriminator", "eatsId", "id", "items", "orderNumber", "paymentInfo", "persons", "platform", "restaurantId", "status"], "type": "object"}` |
| GET | `/api/Integration/v1/order/id:8d853bd485/status` | 200 | confirmed | `{"collections": {}, "fields": ["comment", "status", "updatedAt"], "type": "object"}` |
| GET | `/api/Integration/v1/tableBooking` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/tableBooking` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/tableBooking/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/tableBooking` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/table-booking` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/table-booking` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/table-booking/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/table-booking` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/tableReservation` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/tableReservation` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/tableReservation/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/tableReservation` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/table-reservation` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/table-reservation` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/table-reservation/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/table-reservation` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/booking` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/booking` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/booking/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/booking` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/bookings` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/bookings` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/bookings/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/bookings` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/reservation` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/reservation` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/reservation/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/reservation` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/reservations` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/reservations` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/reservations/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/reservations` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/availability` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/availability` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/availability/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/availability` | 200 | confirmed | `{"collections": {"items": {"count": 0, "item_fields": []}, "modifiers": {"count": 0, "item_fields": []}}, "fields": ["items", "modifiers"], "type": "object"}` |
| GET | `/api/Integration/v1/tableAvailability` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/tableAvailability` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/tableAvailability/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/tableAvailability` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/table-availability` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/table-availability` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/table-availability/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/table-availability` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/bookingAvailability` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/bookingAvailability` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/bookingAvailability/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/bookingAvailability` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/booking-availability` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/booking-availability` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/booking-availability/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/booking-availability` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/table` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/table` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/table/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/table` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/tables` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/tables` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/tables/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/tables` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/hall` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/hall` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/hall/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/hall` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/halls` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/halls` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/halls/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/halls` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/floor` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/floor` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/floor/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/floor` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/floors` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/floors` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/floors/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/floors` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/reserve` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/reserve` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/reserve/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/reserve` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/reserves` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/reserves` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/reserves/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/reserves` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/bron` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/bron` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/bron/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/menu/id:c3f4f1a0d1/bron` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/tableBooking/id:b7fef41433` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/tableBooking/id:b7fef41433` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/tableBooking/id:40aa4c5bcb` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/tableBooking/id:40aa4c5bcb` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/table-booking/id:b7fef41433` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/table-booking/id:b7fef41433` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/table-booking/id:40aa4c5bcb` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/table-booking/id:40aa4c5bcb` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/tableReservation/id:b7fef41433` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/tableReservation/id:b7fef41433` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/tableReservation/id:40aa4c5bcb` | 404 | unsupported | `{"type": "none"}` |
| GET | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/tableReservation/id:40aa4c5bcb` | 404 | unsupported | `{"type": "none"}` |
| OPTIONS | `/api/Integration/v1/tableBooking` | 404 | unsupported | `{"type": "none"}` |
| OPTIONS | `/api/Integration/v1/restaurant/id:c3f4f1a0d1/tableBooking` | 404 | unsupported | `{"type": "none"}` |
| OPTIONS | `/api/Integration/v1/tableBooking/id:c3f4f1a0d1` | 404 | unsupported | `{"type": "none"}` |
| OPTIONS | `/api/Integration/v1/menu/id:c3f4f1a0d1/tableBooking` | 404 | unsupported | `{"type": "none"}` |
| OPTIONS | `/api/Integration/v1/tableBooking/id:b7fef41433` | 404 | unsupported | `{"type": "none"}` |

## Conclusion

At least one non-documented booking-related read route returned usable JSON.
Halls and tables alone do not confirm reservation support.